Plan:
1. Connecting Google Drive and defining paths.
2. Loading the same 61 validation images as those used for Gemini.
3. Verifying the presence of images and masks.
4. Generating 61 comparison images featuring BiomedParse masks (already generated in notebook 5).

In [ ]:
# Import libraries and mount Google Drive.

from pathlib import Path
import pandas as pd
import numpy as np
from PIL import Image

from google.colab import drive
drive.mount("/content/drive")

# Define project directories.
PROJECT_DIR = Path("/content/drive/MyDrive/ovarian_ultrasound_vlm")

DATASET_DIR = Path(
    "/content/drive/MyDrive/biomedparse_project/"
    "nnunet_raw/Dataset001_Follicles"
)

IMAGES_DIR = DATASET_DIR / "imagesTr"
MASKS_DIR = DATASET_DIR / "labelsTr"

MANIFEST_PATH = PROJECT_DIR / "image_manifest.csv"

OUTPUT_DIR = PROJECT_DIR / "evaluation" / "biomedparse_qualitative"
FIGURES_DIR = PROJECT_DIR / "figures" / "biomedparse_qualitative"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

print("Manifest exists:", MANIFEST_PATH.exists())
print("Images directory exists:", IMAGES_DIR.exists())
print("Masks directory exists:", MASKS_DIR.exists())

Mounted at /content/drive
Manifest exists: True
Images directory exists: True
Masks directory exists: True


This notebook evaluates the visual plausibility of BiomedParse pseudo-labels on the same 61 validation images used in the Gemini experiment

In [ ]:
# Define a separate directory for BiomedParse figures.

# Define separate directories for Gemini and BiomedParse figures.

GEMINI_REVIEW_DIR = (
    PROJECT_DIR
    / "figures"
    / "semantic_evaluation"
    / "qualitative_review"
)

REVIEW_FIGURES_DIR = (
    PROJECT_DIR
    / "figures"
    / "biomedparse_qualitative"
    / "qualitative_review"
)

# Prevent mixing the review figures of different models.
assert REVIEW_FIGURES_DIR.resolve() != GEMINI_REVIEW_DIR.resolve()

REVIEW_FIGURES_DIR.mkdir(parents=True, exist_ok=True)

print("Gemini figures:", GEMINI_REVIEW_DIR)
print("BiomedParse figures:", REVIEW_FIGURES_DIR)

Define a separate directory for BiomedParse figures.

In [ ]:
# Load the manifest and select validation images.

manifest = pd.read_csv(MANIFEST_PATH)
# Select the validation split using normalized labels.

split_values = manifest["split"].astype(str).str.strip().str.lower()

validation = manifest.loc[
    split_values.isin(["val", "validation"])
].copy()

print("Validation images:", len(validation))
print(validation["category"].value_counts())

assert len(validation) == 61, (
    f"Expected 61 validation images, found {len(validation)}"
)

print("Validation images:", len(validation))
print(validation["category"].value_counts())

assert len(validation) == 61
assert validation["case_id"].is_unique

# Verify image and pseudo-label paths.
missing_images = []
missing_masks = []

for row in validation.itertuples():
    if not Path(row.image_path).exists():
        missing_images.append(row.case_id)

    if not Path(row.pseudo_label_path).exists():
        missing_masks.append(row.case_id)

print("Missing images:", len(missing_images))
print("Missing masks:", len(missing_masks))

assert not missing_images, missing_images
assert not missing_masks, missing_masks

print("All 61 image-mask pairs are available.")

Validation images: 61
category
dominant    31
pco         21
normal       9
Name: count, dtype: int64
Validation images: 61
category
dominant    31
pco         21
normal       9
Name: count, dtype: int64
Missing images: 0
Missing masks: 0
All 61 image-mask pairs are available.


Use the same validation split as the Gemini evaluation to ensure a consistent comparison

In [ ]:
missing_images = []
missing_masks = []

for row in validation.itertuples():
    if not Path(row.image_path).exists():
        missing_images.append(row.case_id)

    if not Path(row.pseudo_label_path).exists():
        missing_masks.append(row.case_id)

print("Validation images:", len(validation))
print("Missing images:", len(missing_images))
print("Missing masks:", len(missing_masks))

assert len(validation) == 61
assert not missing_images, missing_images
assert not missing_masks, missing_masks

print("All validation image-mask pairs are available.")

Verify all validation image-mask pairs

In [11]:
# Generate large BiomedParse-only qualitative review figures.

import matplotlib.pyplot as plt
import numpy as np
from PIL import Image
from skimage import measure
from pathlib import Path

REVIEW_FIGURES_DIR = (
    PROJECT_DIR / "figures"
    / "biomedparse_qualitative"
    / "qualitative_review"
)

GEMINI_REVIEW_DIR = (
    PROJECT_DIR / "figures"
    / "semantic_evaluation"
    / "qualitative_review"
)

assert REVIEW_FIGURES_DIR.resolve() != GEMINI_REVIEW_DIR.resolve()

REVIEW_FIGURES_DIR.mkdir(parents=True, exist_ok=True)

for _, row in validation.iterrows():

    case_id = row["case_id"]

    # Load the ultrasound image and pseudo-label mask.
    image = np.array(
        Image.open(row["image_path"]).convert("RGB")
    )

    mask = np.array(
        Image.open(row["pseudo_label_path"])
    )

    assert mask.ndim == 2
    assert mask.shape == image.shape[:2]

    binary_mask = mask == 1

    # Count connected components (8-connectivity).
    regions = measure.label(binary_mask, connectivity=2)
    region_count = regions.max()

    # Create the figure.
    fig = plt.figure(figsize=(9, 10))

    gs = fig.add_gridspec(
        nrows=2,
        ncols=1,
        height_ratios=[15, 1],
        hspace=0.02,
        left=0.02,
        right=0.98,
        top=0.94,
        bottom=0.04
    )

    ax = fig.add_subplot(gs[0])
    text_ax = fig.add_subplot(gs[1])

    ax.imshow(image)

    # Add the blue BiomedParse mask.
    overlay = np.zeros(
        (*binary_mask.shape, 4),
        dtype=float
    )

    overlay[binary_mask, :3] = [0.0, 0.5, 1.0]
    overlay[binary_mask, 3] = 0.45

    ax.imshow(overlay)

    ax.set_title(
        "BiomedParse Pseudo-label Overlay",
        fontsize=13
    )
    ax.axis("off")

    # Add a short descriptive caption.
    text_ax.axis("off")

    description = (
        f"BiomedParse identified {region_count} connected "
        f"pseudo-label region(s)."
    )

    text_ax.text(
        0.5, 0.5,
        description,
        ha="center",
        va="center",
        fontsize=13
    )

    fig.suptitle(
        f"{case_id} | BiomedParse count: {region_count}",
        fontsize=13
    )

    output_path = (
        REVIEW_FIGURES_DIR / f"{case_id}_review.png"
    )

    # Preserve already existing images.
    if not output_path.exists():
        fig.savefig(
            output_path,
            dpi=180,
            bbox_inches="tight"
        )

    plt.close(fig)

print(
    "BiomedParse review images:",
    len(list(REVIEW_FIGURES_DIR.glob("*_review.png")))
)

print("Saved to:", REVIEW_FIGURES_DIR)

BiomedParse review images: 61
Saved to: /content/drive/MyDrive/ovarian_ultrasound_vlm/figures/biomedparse_qualitative/qualitative_review


Generate large BiomedParse-only qualitative review figures

In [8]:
# Create a BiomedParse review table using the Gemini review workflow.

import pandas as pd
import numpy as np
from PIL import Image
from skimage import measure
from pathlib import Path

review_records = []

for _, row in validation.iterrows():

    case_id = row["case_id"]

    # Load the BiomedParse pseudo-label mask.
    mask = np.array(
        Image.open(row["pseudo_label_path"])
    )

    assert mask.ndim == 2, f"Expected a 2D mask: {case_id}"

    # Identify connected components using 8-connectivity.
    binary_mask = mask == 1
    labeled_mask = measure.label(binary_mask, connectivity=2)
    regions = measure.regionprops(labeled_mask)

    # Include all region-level geometric measurements.
    region_descriptions = []

    for region_id, region in enumerate(regions, start=1):

        min_row, min_col, max_row, max_col = region.bbox

        region_descriptions.append(
            f"Region {region_id}: "
            f"area={region.area} px; "
            f"bbox=[{min_col}, {min_row}, {max_col}, {max_row}]"
        )

    review_records.append({
        "case_id": case_id,
        "category": row["category"],
        "biomedparse_count": len(regions),
        "region_descriptions": " | ".join(region_descriptions),
        "review_figure": str(
            REVIEW_FIGURES_DIR / f"{case_id}_review.png"
        ),
        "overall_visual_assessment": "",
        "reviewer_notes": ""
    })

review_template = pd.DataFrame(review_records)

# Validate the review table.
assert len(review_template) == 61
assert review_template["case_id"].is_unique

assert review_template["review_figure"].apply(
    lambda path: Path(path).exists()
).all()

# Save the template without overwriting existing manual work.
REVIEW_CSV_PATH = (
    OUTPUT_DIR / "biomedparse_manual_review_template.csv"
)

if REVIEW_CSV_PATH.exists():
    print("Review template already exists; not overwritten.")
    print("Path:", REVIEW_CSV_PATH)
else:
    review_template.to_csv(
        REVIEW_CSV_PATH,
        index=False
    )
    print("Review template saved:", REVIEW_CSV_PATH)

print("Images for review:", len(review_template))

display(review_template.head())

Review template saved: /content/drive/MyDrive/ovarian_ultrasound_vlm/evaluation/biomedparse_qualitative/biomedparse_manual_review_template.csv
Images for review: 61


,case_id,category,biomedparse_count,region_descriptions,review_figure,overall_visual_assessment,reviewer_notes
0,dominant_0011,dominant,1,"Region 1: area=16238.0 px; bbox=[279, 314, 428...",/content/drive/MyDrive/ovarian_ultrasound_vlm/...,,
1,dominant_0012,dominant,1,"Region 1: area=5733.0 px; bbox=[85, 305, 158, ...",/content/drive/MyDrive/ovarian_ultrasound_vlm/...,,
2,dominant_0017,dominant,1,"Region 1: area=16000.0 px; bbox=[93, 293, 262,...",/content/drive/MyDrive/ovarian_ultrasound_vlm/...,,
3,dominant_0019,dominant,1,"Region 1: area=7116.0 px; bbox=[214, 291, 313,...",/content/drive/MyDrive/ovarian_ultrasound_vlm/...,,
4,dominant_0021,dominant,1,"Region 1: area=26438.0 px; bbox=[81, 222, 272,...",/content/drive/MyDrive/ovarian_ultrasound_vlm/...,,


Prepare a review table for the 61 validation images using the same assessment scale as Gemini: Good, Partial, Poor, and Unclear

In [9]:
# Export the BiomedParse review table to a formatted Excel workbook.

from openpyxl import Workbook
from openpyxl.styles import Font, PatternFill, Alignment
from openpyxl.worksheet.datavalidation import DataValidation
from openpyxl.worksheet.table import Table, TableStyleInfo

EXCEL_PATH = OUTPUT_DIR / "biomedparse_manual_review.xlsx"

if EXCEL_PATH.exists():
    print("Workbook already exists; not overwritten:", EXCEL_PATH)

else:
    wb = Workbook()
    ws = wb.active
    ws.title = "BiomedParse review"

    # Add column headers.
    ws.append(review_template.columns.tolist())

    # Add review records.
    for record in review_template.itertuples(index=False, name=None):
        ws.append(list(record))

    # Create a styled Excel table.
    table = Table(
        displayName="BiomedParseReview",
        ref=f"A1:G{ws.max_row}"
    )

    table.tableStyleInfo = TableStyleInfo(
        name="TableStyleMedium2",
        showRowStripes=True,
        showColumnStripes=False
    )

    ws.add_table(table)

    # Configure column widths.
    column_widths = {
        "A": 22,  # case_id
        "B": 16,  # category
        "C": 22,  # biomedparse_count
        "D": 65,  # region_descriptions
        "E": 55,  # review_figure
        "F": 29,  # overall_visual_assessment
        "G": 85   # reviewer_notes
    }

    for column, width in column_widths.items():
        ws.column_dimensions[column].width = width

    # Format rows and enable text wrapping.
    ws.row_dimensions[1].height = 30

    for row in ws.iter_rows(min_row=2):
        ws.row_dimensions[row[0].row].height = 70

        for cell in row:
            cell.alignment = Alignment(
                vertical="top",
                wrap_text=True
            )

    # Add assessment dropdown.
    dropdown = DataValidation(
        type="list",
        formula1='"Good,Partial,Poor,Unclear"',
        allow_blank=True
    )

    dropdown.error = "Choose Good, Partial, Poor, or Unclear."
    dropdown.errorTitle = "Invalid assessment"
    dropdown.showErrorMessage = True

    ws.add_data_validation(dropdown)
    dropdown.add(f"F2:F{ws.max_row}")

    # Freeze header and identifier columns.
    ws.freeze_panes = "C2"

    wb.save(EXCEL_PATH)

    print("Formatted Excel workbook saved:", EXCEL_PATH)

Formatted Excel workbook saved: /content/drive/MyDrive/ovarian_ultrasound_vlm/evaluation/biomedparse_qualitative/biomedparse_manual_review.xlsx


Making a table look nice